In [ ]:
# Empty cell required for the "ruff" extension

# Feature model analysis

Feature models are techniques to extract information from data (in this case, a string).
In the context of near-duplicates detection feature models are used by the Simhash algorithm to create fixed-length fingerprints.
Therefore, it's crucial to find the optimal feature model (and its parameters).
In this project, 4 feature models are considered: bag_of_words, shingles, tf-idf and embeddings.

## bag_of_words

The simplest feature model of the four. For a given string, its features are the words it's made of.
Example: "This is a string" -> ("This", "is", "a", "string")
Hence, two strings are "near-equal" iff they contain about the same words, in any order.

It's simple and quick, but cannot detect locality.
Example: "The bank is after the office" is exactly equal to "The office is after the bank"

## shingles

Strings are sliced using a sliding window algorithm, with a certain _window_size_.
Example: "This is a string" -> ("Thi", "his", " is", "is ", ...) with _window_size_=3
As before, two strings are "near-equal" iff they contain about the same shingles, in any order.

Compared to _bag_of_words_, it generates many more elements but it can detect locality (to some degree).
As a middle-ground, the "word-based" shingle feature model is also considered here: it iterates over words, instead of characters.
Example: "This is a string" -> ("This is", "is a", "a string") with _window_size_=2

## tf-idf

A more refined version of _bag_of_words_, which values "rare" words more than "common" ones.
Each word in each string has a weight (0,1).

## embeddings

Generated using the transformer-based model Qwen3-embedding:0.6b.
Two strings are "near-equal" semantically if their embedding is similar.
Despite its great performance, it tends to be computationally more expensive than other models.


In [ ]:
import sys
from pathlib import Path
from tqdm import tqdm
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.spark_utils import get_spark
import pyspark.sql.functions as F
from src.datasets import read_feature_benchmark
from src.near_duplicates import near_duplicates_benchmark
from src.metrics import ModelMetrics, extract_metrics
from src.fingerprint import FingerprintParams, create_fingerprints
import itertools

%load_ext autoreload
%autoreload 2

In [ ]:
spark = get_spark()
datasets = {
    "easy": read_feature_benchmark(which="easy")
}
datasets["easy"].persist()
datasets["easy"].count()
'''
datasets = {
    "easy": read_feature_benchmark(which="easy"),
    "hard": read_feature_benchmark(which="hard")
}
datasets["easy"].persist()
datasets["easy"].count()
datasets["hard"].persist()
datasets["hard"].count()
'''

FAST_MODE = True
COL_IN1 = "data1"
COL_IN2 = "data2"
COL_OUT1 = "fingerprint1"
COL_OUT2 = "fingerprint2"
TRUTH_COL_SEMANTIC = "is_near_duplicate_semantic"
TRUTH_COL_SYNTAX = "is_near_duplicate_syntax"

In [ ]:
methods = ["bag_of_words", "shingles", "embeddings"] # TODO: add tf-idf
sizes = [64, 128, 256]
Ks = [k for k in range(1, sizes[-1], 5)]

shingle_configs = [
    FingerprintParams(shingle_size=3, over_words=False),
    FingerprintParams(shingle_size=5, over_words=False),
    FingerprintParams(shingle_size=7, over_words=False),
    FingerprintParams(shingle_size=3, over_words=True),
    FingerprintParams(shingle_size=5, over_words=True),
]

experiments = []
for ds, m, s, k in itertools.product(datasets, methods, sizes, Ks):
    if k < s:
        if m == "shingles":
            for config in shingle_configs:
                experiments.append({
                    "dataset": ds,
                    "method": m,
                    "size": s,
                    "K": k,
                    "other": config
                })
        else:
            experiments.append({
                "dataset": ds,
                "method": m,
                "size": s,
                "K": k,
                "other": None
            })

print(f"Combinations to test: {len(experiments)}")

In [ ]:
results = []

for exp in tqdm(experiments, desc="Feature model benchmark running..."):
    # try:
        df_exp = datasets[exp["dataset"]]
        df_exp = create_fingerprints(
            df=df_exp,
            method=exp["method"],
            input_col=COL_IN1,
            output_col=COL_OUT1,
            size=exp["size"],
            fast=FAST_MODE,
            other=exp["other"]
        )
        df_exp = create_fingerprints(
            df=df_exp,
            method=exp["method"],
            input_col=COL_IN2,
            output_col=COL_OUT2,
            size=exp["size"],
            fast=FAST_MODE,
            other=exp["other"]
        )
        metrics = near_duplicates_benchmark(
            df=df_exp,
            f1_col=COL_OUT1,
            f2_col=COL_OUT2,
            truth_col=TRUTH_COL_SEMANTIC,
            K=exp["K"]
        )

        row = {**exp, **metrics.__dict__}
        results.append(row)

    # except Exception as e:
        # print(f"Error during benchmark: {e}")

results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
datasets["easy"].unpersist()
datasets["hard"].unpersist()
spark.stop()